# CARGA DE NUESTROS DATOS

In [156]:
#Importamos nuestras librerías

import pandas as pd
import plotly.express as px

In [157]:
#Procedemos a leer nuestro dataframe y le asignamos el valor df

df = pd.read_csv('retail.csv')

df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [158]:
df.shape

(1000, 9)

In [159]:
#Procedemos a ver los tipos de datos de las columnas

print("--- TIPOS DE DATOS ---")
print(df.info())
print()

print("--- FILAS Y COLUMNAS ---")
print(df.shape)



--- TIPOS DE DATOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB
None

--- FILAS Y COLUMNAS ---
(1000, 9)


In [160]:
#Extraemos estadísticas descriptivas de nuestro Dataframe
df.describe().round(2)

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.00,1000.00,1000.00,1000.00,1000.0
mean,500.50,41.39,2.51,179.89,456.0
std,288.82,13.68,1.13,189.68,560.0
min,1.00,18.00,1.00,25.00,25.0
25%,250.75,29.00,1.00,30.00,60.0
50%,500.50,42.00,3.00,50.00,135.0
75%,750.25,53.00,4.00,300.00,900.0
max,1000.00,64.00,4.00,500.00,2000.0


In [161]:
#Buscamos filas duplicadas o datos vacíos

print("--- DATOS VACÍOS ---")
print(df.isnull().sum())

print("\n--- FILAS DUPLICADAS ---")
print(df.duplicated().sum())

--- DATOS VACÍOS ---
Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

--- FILAS DUPLICADAS ---
0


In [162]:
# BORRAMOS COLUMNAS INNECESARIAS

df.drop(columns=['Customer ID', 'Transaction ID'], inplace=True)



# Vemos nuestros datos nuevamente
df.head()



,Date,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,2023-11-24,Male,34,Beauty,3,50,150
1,2023-02-27,Female,26,Clothing,2,500,1000
2,2023-01-13,Male,50,Electronics,1,30,30
3,2023-05-21,Male,37,Clothing,1,500,500
4,2023-05-06,Male,30,Beauty,2,50,100


In [163]:
#Procedemos a crear el archivo de datos limpios

df.to_csv('Retail_Limpio.csv', index=False)

# FIJAMOS NUESTROS OBJETIVOS

## Impacto de las Caracteristicas del cliente en la Facturación

### Hipotesis

El perfil demográfico de los clientes influye significativamente en el volumen de facturación, existiendo segmentos específicos que generan un mayor ticket promedio que otros.


# Objetivo 1: Determinar si existen diferencias significativas en el ticket promedio entre clientes hombres y mujeres

In [164]:
# Arreglamos nuestros datos para hombres y mujeres

print("--- Total por género ---")
print(df['Gender'].value_counts())

print()
ticket_genero = df.groupby('Gender')['Total Amount'].mean().round(2).reset_index()
print("--- Ticket Promedio por Género ---")
print(ticket_genero)


--- Total por género ---
Gender
Female    510
Male      490
Name: count, dtype: int64

--- Ticket Promedio por Género ---
   Gender  Total Amount
0  Female        456.55
1    Male        455.43


In [165]:
# Gráfico de barras simple
fig1 = px.bar(ticket_genero, 
              x='Gender', 
              y='Total Amount', 
              color='Gender',          
              text_auto='.2f',          
              title='Gasto Promedio por Género',
              color_discrete_map={'Female': 'lightpink', 'Male': 'darkblue'})
fig1.show()

#### Tras analizar las transacciones, descubrimos que el ticket promedio no varía significativamente entre géneros (Mujeres: 456.55 vs Hombres: 455.43). La diferencia es de apenas 1.12. Esto nos indica que el género no es un factor determinante en el monto gastado por visita. Ambos grupos tienen una disposición a gastar prácticamente igual.

# Objetivo 2: Analizar la relación entre la edad del cliente y la categoría de producto elegida

In [166]:
# Creamos una nueva columna clasificadora
df['Grupo Etario'] = 'Mayor a 40'
df.loc[df['Age'] <= 40, 'Grupo Etario'] = '40 o Menor'

# Vemos que la columna se creó correctamente
print("Total de clientes por grupo:")
print(df['Grupo Etario'].value_counts())

print()
# 2. Comparamos el Ticket Promedio usando nuestra nueva columna (¡Un solo groupby en vez de dos variables!)
ticket_edades = df.groupby('Grupo Etario')['Total Amount'].mean().round(2)
print("--- Ticket Promedio por Edad ---")
print(ticket_edades)
print()

Total de clientes por grupo:
Grupo Etario
Mayor a 40    534
40 o Menor    466
Name: count, dtype: int64

--- Ticket Promedio por Edad ---
Grupo Etario
40 o Menor    491.19
Mayor a 40    425.29
Name: Total Amount, dtype: float64



In [167]:
# 2. Calculamos los porcentajes exactamente como tú lo hiciste, pero en una sola tabla
pref_edades = df.groupby('Grupo Etario')['Product Category'].value_counts(normalize=True).reset_index(name='Porcentaje')

# 3. Multiplicamos por 100 para que sea un formato de porcentaje real (ej: 33%)
pref_edades['Porcentaje'] = (pref_edades['Porcentaje'] * 100).round(2)

# Vemos cómo quedó nuestra tabla lista para graficar
pref_edades

,Grupo Etario,Product Category,Porcentaje
0,40 o Menor,Clothing,33.69
1,40 o Menor,Electronics,33.48
2,40 o Menor,Beauty,32.83
3,Mayor a 40,Clothing,36.33
4,Mayor a 40,Electronics,34.83
5,Mayor a 40,Beauty,28.84


In [168]:
# 4. Hacemos el gráfico 
fig2 = px.bar(
    pref_edades,
    x='Product Category',
    y='Porcentaje',
    color='Grupo Etario',
    barmode='group',
    text_auto='.2f',
    title='Preferencias por categoría según grupo de edad',
    labels={'Product Category': 'Categoría'}
)

fig2.show()

 #### La edad tiene un impacto directo en las preferencias de consumo. Los clientes de 40 años o menos son consumidores hacen sus compras de forma equitativa entre Ropa (33.69%), Electrónica (33.48%) y Belleza (32.83%). Mientras que a los clientes de mas de 40 años, pierden interés en la categoría de Belleza (28.84%) y se enfocan en comprar Ropa (36.33%) y Electrónica (34.83%).

# Objetivo 3: Evaluar si la edad del cliente incrementa la tendencia a comprar artículos premium y el gasto total

In [169]:
# CREAMOS RANGOS DE EDAD
df['Rango_Edad'] = 'Por asignar'

df.loc[df['Age'] <= 25, 'Rango_Edad'] = '18-25'
df.loc[(df['Age'] > 25) & (df['Age'] <= 35), 'Rango_Edad'] = '26-35'
df.loc[(df['Age'] > 35) & (df['Age'] <= 50), 'Rango_Edad'] = '36-50'
df.loc[df['Age'] > 50, 'Rango_Edad'] = '51+'


df['Tipo de Producto'] = 'Economico'
df.loc[df['Price per Unit'] > 300, 'Tipo de Producto'] = 'Premium'

analisis = df.groupby(['Rango_Edad', 'Tipo de Producto']).agg({
    'Quantity': 'mean',
    'Total Amount': 'mean',
    'Price per Unit': 'mean'
})

print(analisis)

porcentaje_tipo = (df.groupby('Rango_Edad')['Tipo de Producto']
                     .value_counts(normalize=True)
                     .reset_index(name='Porcentaje'))

porcentaje_tipo['Porcentaje'] = (porcentaje_tipo['Porcentaje'] * 100).round(2)

porcentaje_tipo

                             Quantity  Total Amount  Price per Unit
Rango_Edad Tipo de Producto                                        
18-25      Economico         2.393939    276.893939      105.568182
           Premium           2.594595   1297.297297      500.000000
26-35      Economico         2.664634    286.463415      103.750000
           Premium           2.512195   1256.097561      500.000000
36-50      Economico         2.488281    256.484375      101.933594
           Premium           2.596491   1298.245614      500.000000
51+        Economico         2.526104    238.192771       93.755020
           Premium           2.312500   1156.250000      500.000000


,Rango_Edad,Tipo de Producto,Porcentaje
0,18-25,Economico,78.11
1,18-25,Premium,21.89
2,26-35,Economico,80.00
3,26-35,Premium,20.00
4,36-50,Economico,81.79
5,36-50,Premium,18.21
6,51+,Economico,79.55
7,51+,Premium,20.45


In [170]:
# Usamos px.bar
fig3 = px.bar(
    porcentaje_tipo,                   
    x='Rango_Edad',
    y='Porcentaje',
    color='Tipo de Producto',          
    text_auto='.2f',                   
    title='Proporción de compras: Económico vs Premium por Edad',
    color_discrete_map={'Economico': 'blue', 'Premium': 'red'},
    labels={'Rango_Edad': 'Rango de Edad', 'Tipo de Producto': 'Tipo de Producto'}
)

fig3.show()

##### Los datos demuestran que el segmento más joven (18-25 años) es el que mayor porcentaje de sus compras destina a productos Premium (21.89%). Tambien se observa que en clientes de mayor edad tienen menor preferencia a los articulos premium, siendo el grupo de adultos de 36 a 50 años el que menos artículos de lujo compra (solo 18.21% de Premium frente a un 81.79% de Económicos).


# Objetivo 4: Identificar qué rango de edad concentra el mayor volumen de ingresos acumulados

In [ ]:
analisis = df.groupby('Rango_Edad', as_index=False)['Total Amount'].sum()
analisis


,Rango_Edad,Total Amount
0,18-25,84550
1,26-35,98480
2,36-50,139660
3,51+,133310


In [ ]:
fig4_alt = px.bar(
    analisis, 
    x='Rango_Edad', 
    y='Total Amount', 
    text_auto='.2s',              
    color='Rango_Edad',
    title='Ingresos Acumulados por Rango de Edad'
)

fig4_alt.show()

### Los datos demuestran que el segmento de adultos entre 36 y 50 años aportó el mayor volumen de facturación total ($139,660), seguido por el segmento de personas mayores a 51 años. Tambien podemos apreciar que las personas mas jovenes presentan un menor gasto total. De esta manera identificamos que existe una correlación entre el gasto total y la edad del cliente. 

# Objetivo 5: Evaluar si hombres y mujeres presentan diferencias en el volumen de transacciones y en los ingresos generados

In [ ]:
analisis_segmento = df.groupby(['Rango_Edad', 'Gender'], as_index=False).agg(
    Transacciones=('Total Amount', 'count'),
    Ingresos_Totales=('Total Amount', 'sum')
)

print(analisis_segmento)

,Rango_Edad,Gender,size
0,18-25,Female,81
1,18-25,Male,88
2,26-35,Female,107
3,26-35,Male,98
4,36-50,Female,164
5,36-50,Male,149
6,51+,Female,158
7,51+,Male,155


In [ ]:
fig5_2 = px.bar(
    analisis_segmento,
    x='Rango_Edad',
    y='Ingresos_Totales',
    color='Gender',
    barmode='group',
    title='Ingresos totales por segmento',
    color_discrete_map={
        'Female': 'Pink',
        'Male': 'Blue'
    }
)

fig5_2.show()


### Podemos observar que el segmento de mujeres en el rango de 36-50 años son quienes hacen más transacciones (164) y quienes más dinero dejan ($70,790).